# Construct Q2 model covariates

**Question:** How are recorded covariates constructed for adjusted vitamin D–CRP models?

**Inputs:** The Q1/Q2 cohort, corrected baseline/pre-index CRP and authorized survey/drug/measurement tables.

**Outputs:** Q2 covariate files and availability summaries; the main model selects its stated terms.

Run when the required model covariates need to be built. Some recorded covariates are assessed for availability but do not enter every final model; availability does not imply inclusion.


## Configuration


In [ ]:
from pathlib import Path
import os
import sys

if os.environ.get("CAPSTONE_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ["CAPSTONE_PROJECT_ROOT"]).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "shared" / "q1q2_common.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the example folder, or set CAPSTONE_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / "shared"))
from q1q2_common import *


## Build the paired sample


In [ ]:
base = load_q1_analysis_base()
baseline_crp, _ = run_q2_crp_build(base)
pair = build_q2_pair(base, baseline_crp)


## Covariate definitions and construction


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

_needed = [
    "client",
    "bigquery",
    "CDR",
    "table_path",
    "as_person_id",
    "normalize_index_date",
    "parse_datetime",
    "print_section",
]
_missing = [name for name in _needed if name not in globals()]
if _missing:
    raise RuntimeError("Run Setup and Shared Helper Functions before this cell: " + ", ".join(_missing))

SMOKING_OBS_CONCEPTS = (4282779, 43054909)
ALCOHOL_OBS_CONCEPTS = (45766930, 45772705)

INCOME_CONCEPT_ID = 46235933
EDUCATION_CONCEPT_ID = 40771091
INSURANCE_COVERAGE_CONCEPT_ID = 40766240
HEALTHCARE_ACCESS_BARRIER_CONCEPTS = (
    43530268,  # Delayed Medical Care: Rural Area
    43528664,  # Can't Afford Care: Healthcare Provider
    43530415,  # Can't Afford Care: Delayed Filling Rx To Save Money
    43530594,  # Delayed Medical Care: Nervous
    43529904,  # Delayed Medical Care: Elderly Care
    43529906,  # Delayed Medical Care: Transportation
    43530583,  # Delayed Medical Care: Can't Afford Co-pay
    43529903,  # Delayed Medical Care: Child Care
    43529905,  # Delayed Medical Care: Time Off Work
    43530584,  # Delayed Medical Care: Had To Pay Out Of Pocket
    43530585,  # Delayed Medical Care: Deductible Too High
)

CREATININE_SERUM_PLASMA_CONCEPT = 3016723
AST_CONCEPT_ID = 3013721
BILIRUBIN_TOTAL_CONCEPT_ID = 3024128
ALT_CONCEPT_ID = 3006923
LIVER_LAB_CONCEPTS = {
    "ast": AST_CONCEPT_ID,
    "alt": ALT_CONCEPT_ID,
    "bilirubin_total": BILIRUBIN_TOTAL_CONCEPT_ID,
}


def q2_id_list(pair: pd.DataFrame) -> list[int]:
    values = pd.to_numeric(pair["person_id"], errors="coerce").dropna().astype("int64")
    return [int(x) for x in values.unique().tolist()]


def query_ids(sql: str, ids: list[int]) -> pd.DataFrame:
    job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("ids", "INT64", ids)])
    df = client.query(sql, job_config=job_config).to_dataframe()
    if "person_id" in df.columns:
        df = as_person_id(df)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def q2_dates(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    keep = [col for col in ["person_id", "index_date", "age_at_index", "gender", "female"] if col in base.columns]
    dates = pair[["person_id", "days_before", "crp_mgL"]].merge(base[keep], on="person_id", how="left")
    dates = normalize_index_date(dates)
    dates["days_before"] = pd.to_numeric(dates["days_before"], errors="coerce")
    dates["crp_date"] = dates["index_date"] - pd.to_timedelta(dates["days_before"], unit="D")
    dates["target_date"] = dates["crp_date"].combine_first(dates["index_date"])
    return dates


def write_empty(filename: str, columns: list[str], reason: str) -> pd.DataFrame:
    out = pd.DataFrame(columns=columns)
    out.to_csv(table_path(filename), index=False)
    print(f"{filename}: no rows written ({reason})")
    return out


def nearest_to_target(records: pd.DataFrame, dates: pd.DataFrame, dt_col: str) -> pd.DataFrame:
    if records.empty:
        return records
    d = records.merge(dates[["person_id", "target_date"]], on="person_id", how="left")
    d[dt_col] = parse_datetime(d[dt_col])
    d["gap_days"] = (d[dt_col] - d["target_date"]).abs().dt.days
    return d.sort_values(["person_id", "gap_days"]).groupby("person_id", as_index=False).first()


def summarize_observation_concepts(ids: list[int], concept_ids: list[int], output_file: str) -> pd.DataFrame:
    if not ids or not concept_ids:
        return write_empty(output_file, ["concept_id", "question", "n_rows", "n_people", "n_answer_concepts"], "empty ids/concepts")
    concept_list = ",".join(map(str, concept_ids))
    sql = f"""
    SELECT o.observation_concept_id AS concept_id,
           q.concept_name AS question,
           COUNT(*) AS n_rows,
           COUNT(DISTINCT o.person_id) AS n_people,
           COUNT(DISTINCT o.value_as_concept_id) AS n_answer_concepts
    FROM `{CDR}.observation` o
    JOIN `{CDR}.concept` q ON o.observation_concept_id = q.concept_id
    WHERE o.person_id IN UNNEST(@ids)
      AND o.observation_concept_id IN ({concept_list})
      AND o.value_as_concept_id IS NOT NULL
    GROUP BY 1, 2
    ORDER BY n_people DESC, n_rows DESC
    """
    out = query_ids(sql, ids)
    out.to_csv(table_path(output_file), index=False)
    return out


def fetch_observation_answers(ids: list[int], concept_ids: list[int]) -> pd.DataFrame:
    if not ids or not concept_ids:
        return pd.DataFrame()
    concept_list = ",".join(map(str, concept_ids))
    sql = f"""
    SELECT o.person_id,
           COALESCE(CAST(o.observation_datetime AS TIMESTAMP), TIMESTAMP(o.observation_date)) AS observation_datetime,
           o.observation_concept_id,
           q.concept_name AS question,
           o.value_as_concept_id,
           v.concept_name AS answer
    FROM `{CDR}.observation` o
    LEFT JOIN `{CDR}.concept` q ON o.observation_concept_id = q.concept_id
    LEFT JOIN `{CDR}.concept` v ON o.value_as_concept_id = v.concept_id
    WHERE o.person_id IN UNNEST(@ids)
      AND o.observation_concept_id IN ({concept_list})
      AND o.value_as_concept_id IS NOT NULL
    """
    return query_ids(sql, ids)


def build_q2_smoking_status(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    ids = q2_id_list(pair)
    if not ids:
        return write_empty("q2_smoking_status.csv", ["person_id", "smk_status"], "empty Q2 pair")
    concepts = ",".join(map(str, SMOKING_OBS_CONCEPTS))
    sql = f"""
    SELECT o.person_id,
           COALESCE(CAST(o.observation_datetime AS TIMESTAMP), TIMESTAMP(o.observation_date)) AS observation_datetime,
           o.observation_concept_id,
           q.concept_name AS smoking_question,
           o.value_as_concept_id,
           v.concept_name AS smoking_answer,
           CASE
             WHEN o.value_as_concept_id IN (4144272, 45879404, 45883458, 4222303, 903653, 903651) THEN 'never'
             WHEN o.value_as_concept_id IN (4310250) THEN 'former'
             WHEN o.value_as_concept_id IN (4276526, 42709996, 45881517, 45884037, 37395605, 45881518,
                                            903657, 45878118, 762498, 45884038, 762499, 903659,
                                            4298794, 4034855, 4219336, 903654, 4188539) THEN 'current'
             ELSE NULL
           END AS smk_status
    FROM `{CDR}.observation` o
    LEFT JOIN `{CDR}.concept` q ON o.observation_concept_id = q.concept_id
    LEFT JOIN `{CDR}.concept` v ON o.value_as_concept_id = v.concept_id
    WHERE o.person_id IN UNNEST(@ids)
      AND o.observation_concept_id IN ({concepts})
      AND o.value_as_concept_id IS NOT NULL
    """
    raw = query_ids(sql, ids)
    raw.to_csv(table_path("q2_smoking_status_long.csv"), index=False)
    raw = raw.dropna(subset=["smk_status"])
    if raw.empty:
        return write_empty("q2_smoking_status.csv", ["person_id", "smk_status", "smoking_answer"], "no mapped smoking status")
    out = nearest_to_target(raw, q2_dates(base, pair), "observation_datetime")
    cols = ["person_id", "smk_status", "smoking_answer", "smoking_question", "observation_datetime", "observation_concept_id", "value_as_concept_id"]
    out[cols].to_csv(table_path("q2_smoking_status.csv"), index=False)
    print(f"q2_smoking_status.csv: {len(out):,} participants")
    return out[cols]


def map_alcohol_row(question: object, answer: object) -> str | None:
    qtext = str(question).lower()
    atext = str(answer).lower()
    if "denies alcohol use" in qtext:
        return "none_or_never"
    if "admits alcohol use" in qtext:
        return "current"
    if any(k in atext for k in ["never", "no alcohol", "not at all", "none of the time"]):
        return "none_or_never"
    if any(k in atext for k in ["former", "used to", "previous", "quit"]):
        return "former"
    if any(k in atext for k in ["daily", "weekly", "monthly", "some days", "every day", "yes"]):
        return "current"
    return None


def build_q2_alcohol_use(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    ids = q2_id_list(pair)
    if not ids:
        return write_empty("q2_alcohol_use.csv", ["person_id", "alcohol_use"], "empty Q2 pair")
    summarize_observation_concepts(ids, list(ALCOHOL_OBS_CONCEPTS), "q2_alcohol_use_concept_candidates.csv")
    raw = fetch_observation_answers(ids, list(ALCOHOL_OBS_CONCEPTS))
    if raw.empty:
        return write_empty("q2_alcohol_use.csv", ["person_id", "alcohol_use", "alcohol_answer"], "no alcohol records")
    raw = raw.rename(columns={"question": "alcohol_question", "answer": "alcohol_answer"})
    raw["alcohol_use"] = raw.apply(lambda row: map_alcohol_row(row["alcohol_question"], row["alcohol_answer"]), axis=1)
    raw.to_csv(table_path("q2_alcohol_use_long.csv"), index=False)
    raw = raw.dropna(subset=["alcohol_use"])
    if raw.empty:
        return write_empty("q2_alcohol_use.csv", ["person_id", "alcohol_use", "alcohol_answer"], "no mapped alcohol status")
    out = nearest_to_target(raw, q2_dates(base, pair), "observation_datetime")
    cols = ["person_id", "alcohol_use", "alcohol_answer", "alcohol_question", "observation_datetime", "observation_concept_id", "value_as_concept_id"]
    out[cols].to_csv(table_path("q2_alcohol_use.csv"), index=False)
    print(f"q2_alcohol_use.csv: {len(out):,} participants")
    return out[cols]


def map_income(answer: object) -> str | None:
    text = str(answer).lower()
    if any(k in text for k in ["less 10", "less than 10", "10k 25", "25k 35"]):
        return "Income <35k"
    if any(k in text for k in ["35k 50", "50k 75", "75k 100"]):
        return "Income 35-100k"
    if any(k in text for k in ["100k 150", "150k 200", "more 200", "200k"]):
        return "Income >=100k"
    return None


def map_education(answer: object) -> str | None:
    text = str(answer).lower()
    if any(k in text for k in ["never attended", "kindergarten", "elementary", "junior high", "nine through eleven", "9th", "10th", "11th"]):
        return "Less than high school"
    if "high school" in text or "ged" in text:
        return "High school/GED"
    if "vocational" in text or "technical" in text:
        return "Some college/vocational"
    if "university" in text or "higher education" in text or "college education" in text:
        return "College or higher"
    return None


def map_insurance_coverage(answer: object) -> str | None:
    text = str(answer).lower()
    if any(k in text for k in ["pmi: skip", "prefer not", "don't know", "dont know"]):
        return None
    if any(k in text for k in ["i don't have health insurance", "do not have health insurance", "self-pay", "uninsured"]):
        return "Uninsured"
    if text.strip() == "no":
        return "Uninsured"
    insured_terms = [
        "yes",
        "medicare",
        "medicaid",
        "employer",
        "union",
        "insurance purchased",
        "purchased directly",
        "tricare",
        "military",
        "veterans affairs",
        " va ",
        "government-assistance",
        "other type of health insurance",
        "health coverage plan",
        "indian health service",
    ]
    if any(k in f" {text} " for k in insured_terms):
        return "Insured"
    return None


def map_barrier_answer(answer: object) -> int | None:
    text = str(answer).lower()
    if any(k in text for k in ["pmi: skip", "prefer not", "don't know", "dont know"]):
        return None
    if text.strip() == "yes":
        return 1
    if text.strip() == "no":
        return 0
    return None


def build_fixed_observation_covariate(
    base: pd.DataFrame,
    pair: pd.DataFrame,
    output_file: str,
    target_col: str,
    concept_ids: list[int],
    mapper,
    summary_file: str,
) -> pd.DataFrame:
    ids = q2_id_list(pair)
    if not ids:
        return write_empty(output_file, ["person_id", target_col], "empty Q2 pair")
    summarize_observation_concepts(ids, concept_ids, summary_file)
    raw = fetch_observation_answers(ids, concept_ids)
    if raw.empty:
        return write_empty(output_file, ["person_id", target_col, f"{target_col}_answer"], "no records for fixed concept")
    raw[target_col] = raw["answer"].apply(mapper)
    stem = output_file.replace(".csv", "")
    raw.to_csv(table_path(f"{stem}_long.csv"), index=False)
    raw = raw.dropna(subset=[target_col])
    if raw.empty:
        return write_empty(output_file, ["person_id", target_col, f"{target_col}_answer"], "answers did not map to usable categories")
    out = nearest_to_target(raw, q2_dates(base, pair), "observation_datetime")
    out = out[["person_id", target_col, "answer", "question", "observation_datetime", "observation_concept_id", "value_as_concept_id"]]
    out = out.rename(columns={"answer": f"{target_col}_answer", "question": f"{target_col}_question"})
    out.to_csv(table_path(output_file), index=False)
    print(f"{output_file}: {len(out):,} participants")
    return out


def build_q2_healthcare_access(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    ids = q2_id_list(pair)
    if not ids:
        return write_empty("q2_healthcare_access.csv", ["person_id", "healthcare_access_barrier"], "empty Q2 pair")
    concept_ids = list(HEALTHCARE_ACCESS_BARRIER_CONCEPTS)
    summarize_observation_concepts(ids, concept_ids, "q2_healthcare_access_concept_candidates.csv")
    raw = fetch_observation_answers(ids, concept_ids)
    if raw.empty:
        return write_empty("q2_healthcare_access.csv", ["person_id", "healthcare_access_barrier"], "no access-barrier records")
    raw["healthcare_access_barrier"] = raw["answer"].apply(map_barrier_answer)
    raw.to_csv(table_path("q2_healthcare_access_long.csv"), index=False)
    raw = raw.dropna(subset=["healthcare_access_barrier"])
    if raw.empty:
        return write_empty("q2_healthcare_access.csv", ["person_id", "healthcare_access_barrier"], "no mapped access-barrier answers")
    out = raw.groupby("person_id", as_index=False)["healthcare_access_barrier"].max()
    out.to_csv(table_path("q2_healthcare_access.csv"), index=False)
    print(f"q2_healthcare_access.csv: {len(out):,} participants")
    return out


def build_q2_ses_covariates(base: pd.DataFrame, pair: pd.DataFrame) -> dict[str, pd.DataFrame]:
    income = build_fixed_observation_covariate(
        base, pair, "q2_income_band.csv", "income_band", [INCOME_CONCEPT_ID], map_income, "q2_income_band_concept_candidates.csv"
    )
    education = build_fixed_observation_covariate(
        base, pair, "q2_education.csv", "education_level", [EDUCATION_CONCEPT_ID], map_education, "q2_education_concept_candidates.csv"
    )
    insurance = build_fixed_observation_covariate(
        base, pair, "q2_insurance.csv", "insurance_status", [INSURANCE_COVERAGE_CONCEPT_ID], map_insurance_coverage, "q2_insurance_concept_candidates.csv"
    )
    access = build_q2_healthcare_access(base, pair)
    return {"income": income, "education": education, "insurance": insurance, "healthcare_access": access}


def ckd_epi_2021(scr: float, age: float, female: bool) -> float:
    kappa = 0.7 if female else 0.9
    alpha = -0.241 if female else -0.302
    ratio = scr / kappa
    return 142 * (min(ratio, 1) ** alpha) * (max(ratio, 1) ** -1.200) * (0.9938 ** age) * (1.012 if female else 1.0)


def build_q2_egfr_ckd(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    ids = q2_id_list(pair)
    if not ids:
        return write_empty("q2_egfr_baseline.csv", ["person_id", "egfr", "baseline_ckd"], "empty Q2 pair")
    sql = f"""
    SELECT person_id, measurement_datetime AS dt, value_as_number AS creatinine
    FROM `{CDR}.measurement`
    WHERE person_id IN UNNEST(@ids)
      AND measurement_concept_id = {CREATININE_SERUM_PLASMA_CONCEPT}
      AND value_as_number BETWEEN 0.2 AND 20
    """
    cr = query_ids(sql, ids)
    if cr.empty:
        return write_empty("q2_egfr_baseline.csv", ["person_id", "egfr", "baseline_ckd"], "no creatinine records")
    dates = q2_dates(base, pair)
    cr = cr.merge(dates[["person_id", "index_date"]], on="person_id", how="left")
    cr["dt"] = parse_datetime(cr["dt"])
    cr["days_before_index"] = (cr["index_date"] - cr["dt"]).dt.days
    cr = cr[cr["days_before_index"] >= 0].copy()
    if cr.empty:
        return write_empty("q2_egfr_baseline.csv", ["person_id", "egfr", "baseline_ckd"], "no pre-index creatinine records")
    nearest = cr.sort_values(["person_id", "days_before_index"]).groupby("person_id", as_index=False).first()
    covars = base[[col for col in ["person_id", "age_at_index", "gender", "female"] if col in base.columns]].copy()
    nearest = nearest.merge(covars, on="person_id", how="left")
    if "female" not in nearest.columns and "gender" in nearest.columns:
        nearest["female"] = (nearest["gender"] == "Female").astype(int)
    nearest = nearest.dropna(subset=["creatinine", "age_at_index", "female"])
    nearest["egfr"] = nearest.apply(lambda row: ckd_epi_2021(float(row["creatinine"]), float(row["age_at_index"]), bool(row["female"] == 1)), axis=1)
    nearest["baseline_ckd"] = (nearest["egfr"] < 60).astype(int)
    out = nearest[["person_id", "egfr", "baseline_ckd", "creatinine", "dt", "days_before_index"]]
    out.to_csv(table_path("q2_egfr_baseline.csv"), index=False)
    print(f"q2_egfr_baseline.csv: {len(out):,} participants")
    return out


VITD_SUPPLEMENT_LOOKBACK_DAYS = 120

VITD_SUPPLEMENT_TERMS = (
    "vitamin d",
    "vitamin d2",
    "vitamin d3",
    "cholecalciferol",
    "ergocalciferol",
    "calcifediol",
)

ACTIVE_VITD_ANALOG_TERMS = (
    "calcitriol",
    "paricalcitol",
    "doxercalciferol",
)

LOCAL_ROUTE_TERMS = (
    "topical",
    "cutaneous",
    "cream",
    "ointment",
    "gel",
    "lotion",
    "patch",
    "ophthalmic",
    "otic",
    "nasal",
    "intranasal",
    "inhalation",
    "inhaler",
    "rectal",
    "vaginal",
)

ORAL_RELEVANT_TERMS = (
    "oral",
    "tablet",
    "capsule",
    "chewable",
    "solution",
    "drops",
    "drop",
    "liquid",
    "suspension",
)


def classify_vitd_drug_name(text: object) -> dict[str, int | str]:
    lower = str(text).lower()
    has_supplement_term = any(term in lower for term in VITD_SUPPLEMENT_TERMS)
    has_active_analog_term = any(term in lower for term in ACTIVE_VITD_ANALOG_TERMS)
    has_local_route = any(term in lower for term in LOCAL_ROUTE_TERMS)
    has_oral_relevant_route = any(term in lower for term in ORAL_RELEVANT_TERMS)
    route_class = "local_excluded" if has_local_route else ("oral_or_relevant" if has_oral_relevant_route else "unspecified_relevant")
    return {
        "vitd_supplement_name_match": int(has_supplement_term),
        "active_vitd_analog_name_match": int(has_active_analog_term),
        "local_route_excluded": int(has_local_route),
        "oral_or_relevant_route": int(not has_local_route and (has_oral_relevant_route or has_supplement_term)),
        "vitd_supplement_product": int(has_supplement_term and not has_local_route),
        "active_vitd_analog_product": int(has_active_analog_term and not has_local_route),
        "vitd_route_class": route_class,
    }


def add_drug_end_from_days_supply(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["drug_start"] = parse_datetime(df["drug_start"])
    df["drug_end"] = parse_datetime(df["drug_end"]) if "drug_end" in df.columns else pd.NaT
    df["days_supply"] = pd.to_numeric(df.get("days_supply"), errors="coerce")
    implied_end = df["drug_start"] + pd.to_timedelta(df["days_supply"].clip(lower=1) - 1, unit="D")
    df["drug_end_imputed_from_days_supply"] = df["drug_end"].isna() & df["days_supply"].notna() & df["drug_start"].notna()
    df["drug_end"] = df["drug_end"].combine_first(implied_end)
    return df


def build_q2_medication_use(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    ids = q2_id_list(pair)
    cols = [
        "person_id",
        "vitd_supplement_use",
        "vitd_supplement_120d",
        "vitd_supplement_active_on_target",
        "vitd_supplement_history_pre_target",
        "active_vitd_analog_use_120d",
    ]
    if not ids:
        return write_empty("q2_medication_use.csv", cols, "empty Q2 pair")
    name_condition = " OR ".join([
        "LOWER(drug.concept_name) LIKE '%vitamin d%'", "LOWER(drug.concept_name) LIKE '%cholecalciferol%'",
        "LOWER(drug.concept_name) LIKE '%ergocalciferol%'", "LOWER(drug.concept_name) LIKE '%calcifediol%'",
        "LOWER(drug.concept_name) LIKE '%calcitriol%'", "LOWER(drug.concept_name) LIKE '%paricalcitol%'",
        "LOWER(drug.concept_name) LIKE '%doxercalciferol%'",
    ])
    sql = f"""
    SELECT de.person_id,
           COALESCE(CAST(de.drug_exposure_start_datetime AS TIMESTAMP), TIMESTAMP(de.drug_exposure_start_date)) AS drug_start,
           COALESCE(CAST(de.drug_exposure_end_datetime AS TIMESTAMP), TIMESTAMP(de.drug_exposure_end_date)) AS drug_end,
           de.days_supply,
           de.drug_concept_id,
           drug.concept_name AS drug_name
    FROM `{CDR}.drug_exposure` de
    LEFT JOIN `{CDR}.concept` drug ON de.drug_concept_id = drug.concept_id
    WHERE de.person_id IN UNNEST(@ids)
      AND ({name_condition})
    """
    raw = query_ids(sql, ids)
    if raw.empty:
        return write_empty("q2_medication_use.csv", cols, "no matching drug exposure records")
    dates = q2_dates(base, pair)
    raw = raw.merge(dates[["person_id", "target_date"]], on="person_id", how="left")
    raw = add_drug_end_from_days_supply(raw)
    raw = raw[(raw["drug_start"].isna()) | (raw["target_date"].isna()) | (raw["drug_start"] <= raw["target_date"])].copy()
    if raw.empty:
        return write_empty("q2_medication_use.csv", cols, "no pre-target vitamin D drug exposure records")
    flags = raw["drug_name"].fillna("").apply(classify_vitd_drug_name).apply(pd.Series)
    raw = pd.concat([raw.reset_index(drop=True), flags.reset_index(drop=True)], axis=1)
    raw["days_before_target"] = (raw["target_date"] - raw["drug_start"]).dt.days
    raw["within_120d"] = raw["days_before_target"].between(0, VITD_SUPPLEMENT_LOOKBACK_DAYS, inclusive="both")
    raw["active_on_target"] = (
        raw["drug_start"].notna()
        & raw["target_date"].notna()
        & raw["drug_end"].notna()
        & (raw["drug_start"] <= raw["target_date"])
        & (raw["drug_end"] >= raw["target_date"])
    )
    raw["vitd_supplement_120d"] = (raw["vitd_supplement_product"].eq(1) & raw["within_120d"]).astype(int)
    raw["vitd_supplement_active_on_target"] = (raw["vitd_supplement_product"].eq(1) & raw["active_on_target"]).astype(int)
    raw["vitd_supplement_use"] = (
        raw["vitd_supplement_120d"].eq(1) | raw["vitd_supplement_active_on_target"].eq(1)
    ).astype(int)
    raw["vitd_supplement_history_pre_target"] = raw["vitd_supplement_product"].eq(1).astype(int)
    raw["active_vitd_analog_use_120d"] = (
        raw["active_vitd_analog_product"].eq(1) & (raw["within_120d"] | raw["active_on_target"])
    ).astype(int)
    drug_counts = (
        raw.groupby(["drug_concept_id", "drug_name", "vitd_route_class"], dropna=False)
        .agg(
            n_rows=("person_id", "size"),
            n_people=("person_id", "nunique"),
            vitd_supplement_use=("vitd_supplement_use", "max"),
            vitd_supplement_120d=("vitd_supplement_120d", "max"),
            vitd_supplement_active_on_target=("vitd_supplement_active_on_target", "max"),
            vitd_supplement_history_pre_target=("vitd_supplement_history_pre_target", "max"),
            active_vitd_analog_use_120d=("active_vitd_analog_use_120d", "max"),
            local_route_excluded=("local_route_excluded", "max"),
        )
        .reset_index()
        .sort_values(["n_people", "n_rows"], ascending=False)
    )
    drug_counts.to_csv(table_path("q2_medication_drug_name_counts.csv"), index=False)
    raw.head(5000).to_csv(table_path("q2_medication_use_long_sample.csv"), index=False)
    out = raw.groupby("person_id", as_index=False)[cols[1:]].max()
    out.to_csv(table_path("q2_medication_use.csv"), index=False)
    print(f"q2_medication_use.csv: {len(out):,} participants")
    print(f"vitd_supplement_use uses oral/relevant products within {VITD_SUPPLEMENT_LOOKBACK_DAYS} days before target_date or active on target_date")
    print("Saved q2_medication_drug_name_counts.csv and q2_medication_use_long_sample.csv")
    return out


def build_q2_acute_proxy(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    ids = q2_id_list(pair)
    dates = q2_dates(base, pair)
    out = dates[["person_id", "crp_mgL", "target_date"]].copy()
    out["crp_gt10"] = (pd.to_numeric(out["crp_mgL"], errors="coerce") > 10).astype("Int64")
    out["infection_30d"] = 0
    out["hospitalization_30d"] = 0
    out["ed_visit_30d"] = 0
    if ids:
        infection_condition = " OR ".join([
            "LOWER(c.concept_name) LIKE '%sepsis%'",
            "LOWER(c.concept_name) LIKE '%pneumonia%'",
            "LOWER(c.concept_name) LIKE '%influenza%'",
            "LOWER(c.concept_name) LIKE '%covid%'",
            "LOWER(c.concept_name) LIKE '%cellulitis%'",
            "LOWER(c.concept_name) LIKE '%acute upper respiratory infection%'",
        ])
        infection_sql = f"""
        SELECT co.person_id,
               COALESCE(CAST(co.condition_start_datetime AS TIMESTAMP), TIMESTAMP(co.condition_start_date)) AS condition_start,
               co.condition_concept_id,
               c.concept_name AS condition_name
        FROM `{CDR}.condition_occurrence` co
        LEFT JOIN `{CDR}.concept` c ON co.condition_concept_id = c.concept_id
        WHERE co.person_id IN UNNEST(@ids)
          AND ({infection_condition})
        """
        infections = query_ids(infection_sql, ids)
        if not infections.empty:
            infections = infections.merge(dates[["person_id", "target_date"]], on="person_id", how="left")
            infections["condition_start"] = parse_datetime(infections["condition_start"])
            infections["within_30d"] = (infections["condition_start"] - infections["target_date"]).abs().dt.days <= 30
            true_ids = set(infections.loc[infections["within_30d"], "person_id"].dropna().astype(int))
            out["infection_30d"] = out["person_id"].astype(int).isin(true_ids).astype(int)
            infections.to_csv(table_path("q2_infection_proxy_long.csv"), index=False)
        visit_sql = f"""
        SELECT vo.person_id,
               COALESCE(CAST(vo.visit_start_datetime AS TIMESTAMP), TIMESTAMP(vo.visit_start_date)) AS visit_start,
               vo.visit_concept_id,
               c.concept_name AS visit_name
        FROM `{CDR}.visit_occurrence` vo
        LEFT JOIN `{CDR}.concept` c ON vo.visit_concept_id = c.concept_id
        WHERE vo.person_id IN UNNEST(@ids)
          AND vo.visit_concept_id IN (9201, 9203)
        """
        visits = query_ids(visit_sql, ids)
        if not visits.empty:
            visits = visits.merge(dates[["person_id", "target_date"]], on="person_id", how="left")
            visits["visit_start"] = parse_datetime(visits["visit_start"])
            visits["within_30d"] = (visits["visit_start"] - visits["target_date"]).abs().dt.days <= 30
            hosp_ids = set(visits.loc[(visits["within_30d"]) & (visits["visit_concept_id"] == 9201), "person_id"].dropna().astype(int))
            ed_ids = set(visits.loc[(visits["within_30d"]) & (visits["visit_concept_id"] == 9203), "person_id"].dropna().astype(int))
            out["hospitalization_30d"] = out["person_id"].astype(int).isin(hosp_ids).astype(int)
            out["ed_visit_30d"] = out["person_id"].astype(int).isin(ed_ids).astype(int)
            visits.to_csv(table_path("q2_hospitalization_proxy_long.csv"), index=False)
    final_cols = ["person_id", "crp_gt10", "infection_30d", "hospitalization_30d", "ed_visit_30d"]
    out[final_cols].to_csv(table_path("q2_acute_inflammation_proxy.csv"), index=False)
    print(f"q2_acute_inflammation_proxy.csv: {len(out):,} participants")
    return out[final_cols]


def build_q2_liver_covariates(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    ids = q2_id_list(pair)
    dates = q2_dates(base, pair)
    out = dates[["person_id", "target_date"]].copy()
    out["liver_disease"] = 0
    out["alt"] = np.nan
    out["ast"] = np.nan
    out["bilirubin_total"] = np.nan
    if not ids:
        out.to_csv(table_path("q2_liver_function_disease.csv"), index=False)
        return out
    liver_condition = " OR ".join([
        "LOWER(c.concept_name) LIKE '%liver disease%'",
        "LOWER(c.concept_name) LIKE '%hepatic%'",
        "LOWER(c.concept_name) LIKE '%cirrhosis%'",
        "LOWER(c.concept_name) LIKE '%hepatitis%'",
        "LOWER(c.concept_name) LIKE '%fatty liver%'",
    ])
    condition_sql = f"""
    SELECT co.person_id,
           COALESCE(CAST(co.condition_start_datetime AS TIMESTAMP), TIMESTAMP(co.condition_start_date)) AS condition_start,
           co.condition_concept_id,
           c.concept_name AS condition_name
    FROM `{CDR}.condition_occurrence` co
    LEFT JOIN `{CDR}.concept` c ON co.condition_concept_id = c.concept_id
    WHERE co.person_id IN UNNEST(@ids)
      AND ({liver_condition})
    """
    liver = query_ids(condition_sql, ids)
    if not liver.empty:
        liver = liver.merge(dates[["person_id", "target_date"]], on="person_id", how="left")
        liver["condition_start"] = parse_datetime(liver["condition_start"])
        liver = liver[(liver["condition_start"].isna()) | (liver["target_date"].isna()) | (liver["condition_start"] <= liver["target_date"])]
        out["liver_disease"] = out["person_id"].astype(int).isin(set(liver["person_id"].dropna().astype(int))).astype(int)
        liver.to_csv(table_path("q2_liver_disease_long.csv"), index=False)
    lab_concept_list = ",".join(map(str, LIVER_LAB_CONCEPTS.values()))
    lab_summary_sql = f"""
    SELECT m.measurement_concept_id AS concept_id,
           c.concept_name AS measurement_name,
           COUNT(*) AS n_rows,
           COUNT(DISTINCT m.person_id) AS n_people
    FROM `{CDR}.measurement` m
    JOIN `{CDR}.concept` c ON m.measurement_concept_id = c.concept_id
    WHERE m.person_id IN UNNEST(@ids)
      AND m.measurement_concept_id IN ({lab_concept_list})
      AND m.value_as_number IS NOT NULL
    GROUP BY 1, 2
    ORDER BY n_people DESC, n_rows DESC
    """
    lab_summary = query_ids(lab_summary_sql, ids)
    lab_summary.to_csv(table_path("q2_liver_lab_concept_candidates.csv"), index=False)
    for analyte, concept_id in LIVER_LAB_CONCEPTS.items():
        value_sql = f"""
        SELECT person_id, measurement_datetime AS lab_dt, value_as_number AS value
        FROM `{CDR}.measurement`
        WHERE person_id IN UNNEST(@ids)
          AND measurement_concept_id = {concept_id}
          AND value_as_number IS NOT NULL
        """
        lab = query_ids(value_sql, ids)
        if lab.empty:
            continue
        lab = lab.merge(dates[["person_id", "target_date"]], on="person_id", how="left")
        lab["lab_dt"] = parse_datetime(lab["lab_dt"])
        lab = lab[(lab["lab_dt"].isna()) | (lab["target_date"].isna()) | (lab["lab_dt"] <= lab["target_date"])]
        if lab.empty:
            continue
        lab["gap_days"] = (lab["target_date"] - lab["lab_dt"]).dt.days
        nearest = lab.sort_values(["person_id", "gap_days"]).groupby("person_id", as_index=False).first()
        out = out.drop(columns=[analyte]).merge(nearest[["person_id", "value"]].rename(columns={"value": analyte}), on="person_id", how="left")
    out.to_csv(table_path("q2_liver_function_disease.csv"), index=False)
    print(f"q2_liver_function_disease.csv: {len(out):,} participants")
    return out


def run_q2_candidate_covariate_construction(base: pd.DataFrame, pair: pd.DataFrame, rebuild: bool = False) -> pd.DataFrame:
    print_section("Q2 candidate covariate construction")
    builders = [
        ("smoking", ["q2_smoking_status.csv"], build_q2_smoking_status),
        ("alcohol", ["q2_alcohol_use.csv"], build_q2_alcohol_use),
        ("ses", ["q2_income_band.csv", "q2_education.csv", "q2_insurance.csv", "q2_healthcare_access.csv"], build_q2_ses_covariates),
        ("egfr_ckd", ["q2_egfr_baseline.csv"], build_q2_egfr_ckd),
        ("medication_supplement", ["q2_medication_use.csv", "q2_medication_drug_name_counts.csv"], build_q2_medication_use),
        ("acute_inflammation_proxy", ["q2_acute_inflammation_proxy.csv"], build_q2_acute_proxy),
        ("liver", ["q2_liver_function_disease.csv"], build_q2_liver_covariates),
    ]
    rows = []
    for name, output_files, builder in builders:
        out_paths = [table_path(filename) for filename in output_files]
        primary_output = "; ".join(str(path) for path in out_paths)
        if all(path.exists() for path in out_paths) and not rebuild:
            print(f"{name}: cached")
            rows.append({"construction_step": name, "status": "cached", "primary_output": primary_output, "error": ""})
            continue
        try:
            builder(base, pair)
            rows.append({"construction_step": name, "status": "built", "primary_output": primary_output, "error": ""})
        except Exception as exc:
            print(f"{name} construction failed: {exc}")
            rows.append({"construction_step": name, "status": "failed", "primary_output": primary_output, "error": str(exc)})
    status = pd.DataFrame(rows)
    status.to_csv(table_path("q2_candidate_covariate_construction_status.csv"), index=False)
    print("Saved q2_candidate_covariate_construction_status.csv")
    return status


## Covariate completeness checks


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

_needed = ["OUTPUT_DIR", "table_path", "as_person_id", "normalize_index_date", "print_section"]
_missing = [name for name in _needed if name not in globals()]
if _missing:
    raise RuntimeError("Run Setup and Shared Helper Functions before this cell: " + ", ".join(_missing))


def candidate_file_paths(filenames: list[str]) -> list[Path]:
    paths = []
    for filename in filenames:
        for path in [Path(filename), table_path(filename), OUTPUT_DIR / filename]:
            if path not in paths:
                paths.append(path)
    return paths


def read_optional_person_table(filenames: list[str]) -> tuple[pd.DataFrame | None, str | None]:
    for path in candidate_file_paths(filenames):
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
        except Exception as exc:
            print(f"Optional covariate table skipped ({path}): {exc}")
            continue
        if "person_id" not in df.columns:
            print(f"Optional covariate table skipped ({path}): missing person_id")
            continue
        return as_person_id(df), str(path)
    return None, None


def merge_optional_covariate_table(
    df: pd.DataFrame,
    filenames: list[str],
    column_map: dict[str, str],
    source_key: str,
    sources: dict[str, str],
) -> pd.DataFrame:
    table, source = read_optional_person_table(filenames)
    if table is None:
        return df
    rename = {old: new for old, new in column_map.items() if old in table.columns}
    if not rename:
        print(f"Optional covariate table skipped ({source}): none of {list(column_map)} found")
        return df
    table = table.rename(columns=rename)
    keep = ["person_id"] + sorted(set(rename.values()))
    table = table[keep].drop_duplicates("person_id")
    duplicate_cols = [col for col in keep if col != "person_id" and col in df.columns]
    if duplicate_cols:
        df = df.drop(columns=duplicate_cols)
    sources[source_key] = source or ""
    return df.merge(table, on="person_id", how="left")


def availability_status(source_key: str, sources: dict[str, str], base_status: str) -> tuple[str, str]:
    if source_key == "base":
        return base_status, "analysis base"
    if source_key == "derived":
        return base_status, "derived from Q2 pair/base dates"
    if source_key == "parked":
        return base_status, "parked by current Q2 scope"
    if source_key in sources:
        return "constructed/reusable table found", sources[source_key]
    return "construction output not found", "no readable person-level table found"


def coverage_row(
    df: pd.DataFrame,
    covariate: str,
    fields: list[str],
    source_key: str,
    sources: dict[str, str],
    q2_role: str,
    mode: str = "all",
    base_status: str = "available in analysis base",
    note: str = "",
) -> dict[str, object]:
    present = [field for field in fields if field in df.columns]
    missing = [field for field in fields if field not in df.columns]
    if not present:
        mask = pd.Series(False, index=df.index)
    elif mode == "any":
        mask = df[present].notna().any(axis=1)
    else:
        mask = df[present].notna().all(axis=1)
    status, source = availability_status(source_key, sources, base_status)
    n_available = int(mask.sum())
    denominator = len(df)
    return {
        "covariate": covariate,
        "fields_checked": ", ".join(fields),
        "present_fields": ", ".join(present),
        "missing_fields": ", ".join(missing),
        "availability_status": status,
        "q2_role_now": q2_role,
        "source_or_construction": source,
        "n_available_in_q2_pair": n_available,
        "q2_pair_n": denominator,
        "coverage_pct": 100 * n_available / denominator if denominator else np.nan,
        "note": note,
    }


def build_q2_covariate_level_counts(df: pd.DataFrame, fields: list[str]) -> pd.DataFrame:
    rows = []
    for field in fields:
        if field not in df.columns:
            continue
        counts = df[field].astype("object").where(df[field].notna(), "<missing>").value_counts(dropna=False)
        for value, count in counts.items():
            rows.append({"field": field, "level": str(value), "n": int(count), "pct_of_q2_pair": 100 * int(count) / len(df) if len(df) else np.nan})
    return pd.DataFrame(rows)


def build_q2_covariate_continuous_summary(df: pd.DataFrame, fields: list[str]) -> pd.DataFrame:
    rows = []
    for field in fields:
        if field not in df.columns:
            continue
        values = pd.to_numeric(df[field], errors="coerce")
        rows.append({
            "field": field,
            "n_nonmissing": int(values.notna().sum()),
            "mean": values.mean(),
            "sd": values.std(),
            "p25": values.quantile(0.25),
            "median": values.median(),
            "p75": values.quantile(0.75),
            "min": values.min(),
            "max": values.max(),
        })
    return pd.DataFrame(rows)


def run_q2_covariate_availability_audit(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    base_cols = [
        "person_id", "index_date", "age_at_index", "female", "gender", "bmi_full",
        "race_eth_simplified", "race_black", "race_hispanic", "race_asian", "race_other",
        "baseline_diabetes", "baseline_hypertension", "baseline_hyperlipidemia",
    ]
    audit = pair.merge(base[[col for col in base_cols if col in base.columns]], on="person_id", how="left")
    audit = normalize_index_date(audit)
    audit["days_before"] = pd.to_numeric(audit.get("days_before"), errors="coerce")
    audit["crp_date"] = audit["index_date"] - pd.to_timedelta(audit["days_before"], unit="D")
    audit["blood_draw_month"] = audit["crp_date"].dt.month
    audit["winter"] = audit["blood_draw_month"].isin([11, 12, 1, 2, 3, 4]).astype("Int64")
    audit["crp_gt10"] = (pd.to_numeric(audit["crp_mgL"], errors="coerce") > 10).astype("Int64")

    sources: dict[str, str] = {}
    merge_specs = [
        ("smoking", ["q2_smoking_status.csv", "q1_smoking_status.csv"], {"smk_status": "smk_status", "smoking_status": "smk_status"}),
        ("alcohol", ["q2_alcohol_use.csv", "q1_alcohol_use.csv"], {"alcohol_use": "alcohol_use"}),
        ("egfr_ckd", ["q2_egfr_baseline.csv", "q1_egfr_baseline.csv"], {"egfr": "egfr", "baseline_ckd": "baseline_ckd"}),
        ("income", ["q2_income_band.csv", "q1_income_band.csv", "income_band.csv", "household_income.csv"], {"income_band": "income_band", "g_income": "income_band"}),
        ("education", ["q2_education.csv", "education.csv"], {"education_level": "education_level", "education": "education_level"}),
        ("insurance", ["q2_insurance.csv", "insurance.csv"], {"insurance_status": "insurance_status", "insurance": "insurance_status"}),
        ("healthcare_access", ["q2_healthcare_access.csv", "healthcare_access.csv"], {"healthcare_access_barrier": "healthcare_access_barrier", "healthcare_access": "healthcare_access_barrier"}),
        ("medication", ["q2_medication_use.csv"], {
            "vitd_supplement_use": "vitd_supplement_use",
            "vitd_supplement_120d": "vitd_supplement_120d",
            "vitd_supplement_active_on_target": "vitd_supplement_active_on_target",
            "vitd_supplement_history_pre_target": "vitd_supplement_history_pre_target",
            "active_vitd_analog_use_120d": "active_vitd_analog_use_120d",
        }),
        ("acute_proxy", ["q2_acute_inflammation_proxy.csv"], {"infection_30d": "infection_30d", "hospitalization_30d": "hospitalization_30d", "ed_visit_30d": "ed_visit_30d"}),
        ("liver", ["q2_liver_function_disease.csv"], {"liver_disease": "liver_disease", "alt": "alt", "ast": "ast", "bilirubin_total": "bilirubin_total"}),
    ]
    for key, filenames, cmap in merge_specs:
        audit = merge_optional_covariate_table(audit, filenames, cmap, key, sources)

    current_cols = ["log_crp", "log_vitd", "age_at_index", "female", "winter", "bmi_full"]
    current_mask = audit[current_cols].notna().all(axis=1)
    current_mask = current_mask & pd.to_numeric(audit["bmi_full"], errors="coerce").between(12, 70)

    specs = [
        ("Vitamin D exposure", ["vitd_main", "log_vitd"], "base", "exposure", "all", "already in Q2 pair", ""),
        ("CRP outcome", ["crp_mgL", "log_crp"], "base", "outcome", "all", "already in Q2 pair", ""),
        ("Age", ["age_at_index"], "base", "current covariate", "all", "already in current Q2 model", ""),
        ("Sex", ["female"], "base", "current covariate", "all", "already in current Q2 model", ""),
        ("BMI", ["bmi_full"], "base", "current covariate", "all", "already in current Q2 model", "Q2 OLS applies BMI range filter 12-70"),
        ("Season/month", ["blood_draw_month", "winter"], "derived", "current covariate", "all", "already in current Q2 model", "winter is Nov-Apr"),
        ("Race/ethnicity", ["race_black", "race_hispanic", "race_asian", "race_other"], "base", "available to add", "all", "available in analysis base, not in Q2 model", ""),
        ("Diabetes", ["baseline_diabetes"], "base", "available to add", "all", "available in analysis base, not in Q2 model", ""),
        ("Hypertension", ["baseline_hypertension"], "base", "available to add or sensitivity", "all", "available in analysis base, not in Q2 model", ""),
        ("Hyperlipidemia", ["baseline_hyperlipidemia"], "base", "available to add or sensitivity", "all", "available in analysis base, not in Q2 model", ""),
        ("Smoking", ["smk_status"], "smoking", "candidate covariate", "all", "", "constructed before investigating coverage shortfall"),
        ("Alcohol", ["alcohol_use"], "alcohol", "candidate covariate", "all", "", "constructed before investigating coverage shortfall"),
        ("Kidney function", ["egfr", "baseline_ckd"], "egfr_ckd", "candidate/sensitivity covariate", "any", "", "constructed from creatinine"),
        ("Household income", ["income_band"], "income", "candidate SES covariate", "all", "", "SES component"),
        ("Education", ["education_level"], "education", "candidate SES covariate", "all", "", "SES component"),
        ("Insurance", ["insurance_status"], "insurance", "candidate SES/access covariate", "all", "", "SES/access component"),
        ("Healthcare access", ["healthcare_access_barrier"], "healthcare_access", "candidate SES/access covariate", "all", "", "SES/access component"),
        ("Vitamin D supplement use", ["vitd_supplement_use"], "medication", "candidate covariate", "all", "", "120-day near-target oral/relevant vitamin D supplement definition"),
        ("Vitamin D supplement detail", ["vitd_supplement_120d", "vitd_supplement_active_on_target", "vitd_supplement_history_pre_target"], "medication", "audit detail", "any", "", "documents current/recent versus historical vitamin D supplement records"),
        ("Active vitamin D analog use", ["active_vitd_analog_use_120d"], "medication", "audit detail", "all", "", "not counted as ordinary vitamin D supplement use"),
        ("Any supplement use", ["any_supplement_use"], "parked", "parked", "all", "parked in current supplement-only pass", "current construction only targets vitamin D supplement use"),
        ("Medication use", ["statin_use", "systemic_steroid_use", "nsaid_use"], "parked", "parked", "any", "parked in current supplement-only pass", "steroid/NSAID parked; statin not required for this supplement-focused pass"),
        ("Acute inflammation flag", ["crp_gt10"], "derived", "sensitivity-ready", "all", "derivable from Q2 CRP", ""),
        ("Infection/hospitalization proxy", ["infection_30d", "hospitalization_30d", "ed_visit_30d"], "acute_proxy", "sensitivity candidate", "any", "", "constructed around CRP date"),
        ("Liver function/disease", ["liver_disease", "alt", "ast"], "liver", "sensitivity candidate", "any", "", "diagnosis or lab proxy"),
    ]
    coverage = pd.DataFrame([
        coverage_row(audit, covariate, fields, source_key, sources, role, mode, status, note)
        for covariate, fields, source_key, role, mode, status, note in specs
    ])
    coverage.insert(7, "current_q2_model_complete_case_n", int(current_mask.sum()))

    categorical_fields = [
        "female", "winter", "blood_draw_month", "race_eth_simplified",
        "baseline_diabetes", "baseline_hypertension", "baseline_hyperlipidemia",
        "smk_status", "alcohol_use", "baseline_ckd", "income_band", "education_level",
        "insurance_status", "healthcare_access_barrier", "vitd_supplement_use",
        "vitd_supplement_120d", "vitd_supplement_active_on_target", "vitd_supplement_history_pre_target",
        "active_vitd_analog_use_120d", "any_supplement_use", "statin_use", "systemic_steroid_use", "nsaid_use",
        "crp_gt10", "infection_30d", "hospitalization_30d", "ed_visit_30d", "liver_disease",
    ]
    continuous_fields = ["vitd_main", "crp_mgL", "log_vitd", "log_crp", "age_at_index", "bmi_full", "days_before", "egfr", "alt", "ast", "bilirubin_total"]
    level_counts = build_q2_covariate_level_counts(audit, categorical_fields)
    continuous_summary = build_q2_covariate_continuous_summary(audit, continuous_fields)

    audit.to_csv(table_path("q2_covariate_exploration_dataset.csv"), index=False)
    coverage.to_csv(table_path("q2_covariate_coverage.csv"), index=False)
    level_counts.to_csv(table_path("q2_covariate_level_counts.csv"), index=False)
    continuous_summary.to_csv(table_path("q2_covariate_continuous_summary.csv"), index=False)

    print_section("Q2 covariate availability exploration")
    print(f"Q2 pair n: {len(audit):,}")
    print(f"Current Q2 M4 complete-case n: {int(current_mask.sum()):,}")
    display_cols = ["covariate", "q2_role_now", "n_available_in_q2_pair", "coverage_pct", "availability_status"]
    print(coverage[display_cols].to_string(index=False, float_format=lambda x: f"{x:.1f}"))
    print("Saved q2_covariate_exploration_dataset.csv, q2_covariate_coverage.csv, q2_covariate_level_counts.csv, q2_covariate_continuous_summary.csv")
    return coverage


## Construct and audit


In [ ]:
run_q2_candidate_covariate_construction(base, pair)
run_q2_covariate_availability_audit(base, pair)
